# Vision-Language and Multimodal Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week07_vision_language.ipynb)

*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.*

**BME 6938 Advanced Biomedical AI · Week 7 · Reference notebook (ungraded)**

This notebook follows the Week 7 lecture and its three clips. It reproduces every number in the clips with open models and
open data: the H&E tiles of Kather et al. 2016 (the same tiles as in Week 6), the general CLIP model from OpenAI, BiomedCLIP,
and a small open vision-language model, SmolVLM-256M.

| Section | Clip | What you will see |
|---|---|---|
| 1 | CLIP: matching images with their captions | Two encoders, the 4 × 4 similarity grid, the softmax over rows and columns, the CLIP loss |
| 2 | Zero-shot classification, and where it breaks | Prompts as classifier weights; CLIP and BiomedCLIP on all 5,000 tiles; a linear probe with a few labels |
| 3 | Images as tokens: Med-PaLM M | How a vision-language model turns an image into tokens, and a loss counted only on the report |

**How to run it**
- **Google Colab:** *Runtime > Run all*. On a CPU the notebook takes about 10 minutes plus downloads, most of it in section
  2.5, which embeds all 5,000 tiles with two models. A GPU (*Runtime > Change runtime type > T4 GPU*) makes it faster.
- **HiPerGator or your own computer:** a Jupyter kernel with PyTorch. Set `DATA_DIR` in the setup cell to a folder on your
  group's `/blue` storage.
- The first run downloads the Kather tiles (about 260 MB), CLIP ViT-B/32 (about 350 MB), BiomedCLIP (about 800 MB), and
  SmolVLM-256M (about 500 MB). No account is needed.

**Data and models.** H&E tiles: Kather JN et al., *Multi-class texture analysis in colorectal cancer histology*, Scientific
Reports 6, 27988 (2016), [doi:10.5281/zenodo.53169](https://doi.org/10.5281/zenodo.53169), CC BY 4.0. CLIP: Radford et al.,
ICML 2021, weights released by OpenAI under the MIT license, loaded with [open_clip](https://github.com/mlfoundations/open_clip).
BiomedCLIP: Zhang et al., [microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224](https://huggingface.co/microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224),
MIT license. SmolVLM-256M-Instruct: Hugging Face, Apache 2.0. Med-PaLM M itself is not public, so section 3 uses SmolVLM,
which is built the same way: an image encoder, a projection, and a language model.

## 0. Setup

In [ ]:
import importlib, subprocess, sys
if importlib.util.find_spec("open_clip") is None:              # open_clip loads CLIP and BiomedCLIP
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch"], check=False)

import os, glob, time, zipfile, urllib.request                 # files, timing, download
import numpy as np                                             # arrays
import torch                                                   # tensors
import torch.nn.functional as F                                # softmax and cross-entropy
import matplotlib.pyplot as plt                                # plots
from PIL import Image                                          # reads the tiles
import open_clip                                               # CLIP-style models
from sklearn.linear_model import LogisticRegression            # the linear probe in section 2.7

DATA_DIR = os.path.expanduser("~/bme6938_data")                # on HiPerGator, use a folder on /blue
os.makedirs(DATA_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| device:", DEVICE)

The Kather 2016 tiles: 5,000 tiles of 150 × 150 pixels, 625 in each of 8 tissue classes. The download is the same as in the
Week 6 notebook, so if you ran that one with the same `DATA_DIR`, nothing is downloaded again.

In [ ]:
URL = "https://zenodo.org/api/records/53169/files/Kather_texture_2016_image_tiles_5000.zip/content"
zip_path = os.path.join(DATA_DIR, "Kather_texture_2016_image_tiles_5000.zip")
for attempt in range(3):                                       # download once, retrying if the connection drops
    if os.path.exists(zip_path): break
    try:
        print("downloading the Kather 2016 tiles ...")
        urllib.request.urlretrieve(URL, zip_path + ".part")    # temporary name first ...
        os.replace(zip_path + ".part", zip_path)               # ... so a broken download is never mistaken for the zip
    except OSError as error:
        print("download failed:", error, "| retrying" if attempt < 2 else "| check the connection and run this cell again")
tile_dir = os.path.join(DATA_DIR, "Kather_texture_2016_image_tiles_5000")
if not os.path.isdir(tile_dir):
    with zipfile.ZipFile(zip_path) as z: z.extractall(DATA_DIR)

FOLDERS = sorted(os.listdir(tile_dir))                         # 01_TUMOR ... 08_EMPTY
files, labels = [], []
for k, folder in enumerate(FOLDERS):
    for f in sorted(glob.glob(os.path.join(tile_dir, folder, "*.tif"))):
        files.append(f); labels.append(k)
labels = np.array(labels)
tile = lambda name: Image.open(next(f for f in files if os.path.basename(f) == name)).convert("RGB")   # one tile by file name

# The four tiles drawn in the clips (file names from lectures/_shared/assets/README.md)
CLIP_TILES = {"tumor": "10009_CRC-Prim-HE-03_009.tif_Row_301_Col_151.tif",
              "normal mucosa": "1008A_CRC-Prim-HE-03_007.tif_Row_301_Col_1051.tif",
              "adipose tissue": "10006_CRC-Prim-HE-03_012.tif_Row_601_Col_2251.tif",
              "lymphocytes": "1017A_CRC-Prim-HE-05_022.tif_Row_301_Col_1.tif"}
fig, axes = plt.subplots(1, 4, figsize=(8, 2.3))
for ax, (name, f) in zip(axes, CLIP_TILES.items()):
    ax.imshow(tile(f)); ax.set_title(name, fontsize=9); ax.axis("off")
plt.suptitle("H&E tiles: Kather et al. 2016, CC BY 4.0", fontsize=8); plt.tight_layout(); plt.show()
print(len(files), "tiles in", len(FOLDERS), "classes:", [f[3:].lower() for f in FOLDERS])

## 1. CLIP: matching images with their captions  *(clip 1)*

### 1.1 CLIP pairs an image with its caption, instead of two views of one image
In Week 6, contrastive learning pulled two augmented views of one ECG together. CLIP pulls an **image** and its **caption**
together, and pushes apart the image and every other caption in the batch.

### 1.2 An image encoder and a text encoder map both into one shared embedding space
We load BiomedCLIP. Its image encoder is a vision transformer and its text encoder is PubMedBERT. Both end in a vector of
**512 numbers**, which we scale to length 1.

In [ ]:
def load(name):
    # Returns (model, image preprocessing, tokenizer) for an open_clip model name.
    if name.startswith("hf-hub:"):
        model, preprocess = open_clip.create_model_from_pretrained(name)
        tokenizer = open_clip.get_tokenizer(name)
    else:
        arch, tag = name.split("/")
        model, _, preprocess = open_clip.create_model_and_transforms(arch, pretrained=tag)
        tokenizer = open_clip.get_tokenizer(arch)
    return model.to(DEVICE).eval(), preprocess, tokenizer

BIOMEDCLIP = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
model, preprocess, tokenizer = load(BIOMEDCLIP)

@torch.no_grad()
def embed_images(model, preprocess, images, batch=64):
    # PIL images -> (N, 512) unit vectors.
    out = []
    for i in range(0, len(images), batch):
        x = torch.stack([preprocess(im) for im in images[i:i + batch]]).to(DEVICE)
        out.append(F.normalize(model.encode_image(x), dim=-1).cpu())
    return torch.cat(out)

@torch.no_grad()
def embed_texts(model, tokenizer, texts):
    # Strings -> (N, 512) unit vectors.
    return F.normalize(model.encode_text(tokenizer(texts).to(DEVICE)), dim=-1).cpu()

z_img = embed_images(model, preprocess, [tile(CLIP_TILES["tumor"])])
z_txt = embed_texts(model, tokenizer, ["colorectal adenocarcinoma, H&E"])
print("image vector:", tuple(z_img.shape), "length", round(z_img.norm().item(), 3))
print("text vector: ", tuple(z_txt.shape), "length", round(z_txt.norm().item(), 3))
print("cosine similarity:", round((z_img @ z_txt.T).item(), 3))

### 1.3 A batch of four pairs gives a 4 × 4 grid of similarities; the matching pairs lie on the diagonal
The clip uses an illustrative grid, chosen so the arithmetic is easy to follow. Row i is tile i, column j is caption j.

In [ ]:
NAMES = ["tumor", "normal mucosa", "adipose tissue", "lymphocytes"]
S = torch.tensor([[0.32, 0.18, 0.05, 0.12],                    # the grid from clip 1 (illustrative)
                  [0.20, 0.30, 0.06, 0.10],
                  [0.04, 0.07, 0.34, 0.03],
                  [0.14, 0.11, 0.02, 0.29]])
print(S)

### 1.4 Each row becomes a softmax: the tumor tile gives its own caption 0.82
Divide by the temperature τ = 0.07 and apply a softmax along each row. The first row is the tumor tile choosing among the four
captions.

In [ ]:
tau = 0.07
rows = torch.softmax(S / tau, dim=1)                           # image -> text: each tile picks a caption
cols = torch.softmax(S / tau, dim=0)                           # text -> image: each caption picks a tile
print("tumor tile, probability of each caption:", rows[0].numpy().round(2))   # 0.82 for its own caption

### 1.5 The loss is cross-entropy along every row and every column, averaged
The correct answer for row i (and column i) is i: the diagonal. CLIP averages the image-to-text and the text-to-image
cross-entropy.

In [ ]:
def clip_loss(sim, tau):
    # Symmetric CLIP loss for a square similarity matrix whose diagonal holds the true pairs.
    target = torch.arange(len(sim))                            # the diagonal
    image_to_text = F.cross_entropy(sim / tau, target)         # softmax over each row
    text_to_image = F.cross_entropy(sim.T / tau, target)       # softmax over each column
    return (image_to_text + text_to_image) / 2

print("row losses:", (-torch.log(rows.diag())).numpy().round(2))
print("column losses:", (-torch.log(cols.diag())).numpy().round(2))
print("CLIP loss:", round(clip_loss(S, tau).item(), 2))       # 0.18, as in the clip

**The same grid from the real model.** Here are BiomedCLIP's cosine similarities for the four tiles and four short captions.
The real numbers are closer together than the illustrative ones. In three of the four rows the diagonal is the largest; the
normal mucosa tile is closer to the tumor caption than to its own, the same mistake we will see again in section 2.3. BiomedCLIP does not use τ = 0.07: during training it learned its own temperature, stored as `logit_scale` = 1 / τ.

In [ ]:
Z_img = embed_images(model, preprocess, [tile(f) for f in CLIP_TILES.values()])
Z_txt = embed_texts(model, tokenizer, [f"an H&E image of {n}." for n in NAMES])
real = Z_img @ Z_txt.T                                         # 4 x 4 cosine similarities
print(real.numpy().round(2))
scale = model.logit_scale.exp().item()
print(f"learned logit_scale {scale:.1f}, so tau = {1 / scale:.4f}")
print("CLIP loss on this real batch:", round(clip_loss(real, 1 / scale).item(), 3))

### 1.5b Watch the embedding space sort itself out

Section 1.5 wrote the loss down. The animation runs it, on six synthetic pairs, so you can watch what it does to the
space.

Every encoder output is a unit vector, so the shared embedding space is a circle here rather than the 512-dimensional
sphere CLIP uses. Blue points are tiles, orange points are captions, and a line joins each tile to its own caption. At
the start the pairing is arbitrary and the similarity matrix on the right is a mess. Each step applies the gradient the
loss asks for: pull a pair together, push everything else in the batch apart.

What to notice is that the matrix goes diagonal without anyone ever being told what a tumour looks like. The only
supervision is which caption came with which tile, and that is free: it is how the data was stored in the first place.

In [ ]:
# An animation, drawn by the code below. It renders with to_jshtml, which needs no
# ffmpeg and plays in Colab, in Jupyter, and on nbviewer. Press play, or drag the
# slider to step through it frame by frame.
import numpy as np                                           # arrays
import matplotlib.pyplot as plt                              # figures
from matplotlib.animation import FuncAnimation               # the animation itself
from IPython.display import HTML                             # shows it in the notebook

UF, ORANGE, GREY = "#0021A5", "#FA4616", "#6B7280"           # the course colours
plt.rcParams["figure.dpi"] = 78                              # keeps the rendered animation light

N, STEPS, TEMP = 6, 60, 0.10                                 # pairs in the batch, training steps, temperature
LABELS = ["tumour", "stroma", "lymphocytes", "mucosa", "debris", "adipose"]
rng = np.random.default_rng(1)
img = rng.uniform(0, 2 * np.pi, N)                           # each tile embedding, as an angle on the circle
txt = rng.uniform(0, 2 * np.pi, N)                           # each caption embedding, likewise

def sims(a, b):
    """Cosine similarity between every tile and every caption, which on a circle is cos of the angle between them."""
    return np.cos(a[:, None] - b[None, :])

hist = [(img.copy(), txt.copy())]                            # record the whole run, then animate it
for _ in range(STEPS):
    S = sims(img, txt) / TEMP
    P = np.exp(S - S.max(1, keepdims=True)); P = P / P.sum(1, keepdims=True)   # softmax along each row
    # The InfoNCE gradient, in words: move each caption toward its own tile, and
    # away from every tile that is currently giving it weight it has not earned.
    pull = np.zeros(N)
    for i in range(N):
        for j in range(N):
            d = np.sin(img[i] - txt[j])                      # direction that increases cos(img_i - txt_j)
            pull[j] += (1.0 if i == j else 0.0) * d - P[i, j] * d
    txt = (txt + 0.45 * pull) % (2 * np.pi)
    img = (img - 0.18 * np.array([np.sin(img[i] - txt[i]) for i in range(N)])) % (2 * np.pi)
    hist.append((img.copy(), txt.copy()))

fig, (circ, mat) = plt.subplots(1, 2, figsize=(9.2, 3.9),
                                gridspec_kw={"width_ratios": [1.1, 1]})
fig.subplots_adjust(wspace=0.35)

def draw(f):
    a, b = hist[f]
    circ.clear(); mat.clear()

    # ---- left: the shared embedding space
    th = np.linspace(0, 2 * np.pi, 200)
    circ.plot(np.cos(th), np.sin(th), color="#E5E7EB", lw=1.2)
    for i in range(N):
        circ.plot([np.cos(a[i]), np.cos(b[i])], [np.sin(a[i]), np.sin(b[i])],
                  color="#C7CDD1", lw=1.0, zorder=0)         # the line joining a true pair
    circ.scatter(np.cos(a), np.sin(a), s=70, color=UF, zorder=3, label="tile")
    circ.scatter(np.cos(b), np.sin(b), s=70, color=ORANGE, zorder=3, marker="s", label="caption")
    for i in range(N):
        circ.annotate(LABELS[i], (np.cos(a[i]) * 1.17, np.sin(a[i]) * 1.17),
                      ha="center", va="center", fontsize=8.5, color="#374151")
    circ.set_xlim(-1.45, 1.45); circ.set_ylim(-1.45, 1.45)
    circ.set_aspect("equal"); circ.axis("off")
    circ.legend(loc="lower right", fontsize=9, frameon=False)
    circ.set_title(f"step {f} of {STEPS}", loc="left", fontsize=11, color="#374151")

    # ---- right: the similarity matrix the loss actually sees
    S = sims(a, b)
    mat.imshow(S, cmap="Blues", vmin=-1, vmax=1)
    mat.set_xticks(range(N)); mat.set_xticklabels(LABELS, fontsize=8, rotation=40, ha="right")
    mat.set_yticks(range(N)); mat.set_yticklabels(LABELS, fontsize=8)
    mat.set_xlabel("caption", fontsize=10); mat.set_ylabel("tile", fontsize=10)
    acc = (S.argmax(1) == np.arange(N)).mean()               # how many rows pick their own caption
    mat.set_title(f"rows picking their own caption: {acc * 100:.0f}%",
                  loc="left", fontsize=10, color=GREY)
    return []

ani = FuncAnimation(fig, draw, frames=len(hist), interval=90)
plt.close(fig)
HTML(ani.to_jshtml(fps=11))

### 1.6 and 1.7 Scale
CLIP was trained on 400 million image-text pairs from the web, in batches of 32,768 pairs (a 32,768 × 32,768 grid), with a
temperature that starts at 0.07 and is learned (Radford et al., 2021). BiomedCLIP uses the same recipe on 15 million
figure-caption pairs from 4.4 million PubMed Central articles (Zhang et al.).

## 2. Zero-shot classification, and where it breaks  *(clip 2)*

### 2.1 Zero-shot classification starts with one text prompt per class
### 2.2 The text encoder turns the prompts into class vectors: a linear classifier written in text

In [ ]:
CLASSES = ["colorectal adenocarcinoma", "tumor stroma", "complex stroma", "lymphocytes",
           "necrotic debris", "normal colon mucosa", "adipose tissue", "background"]   # one per Kather folder, in order
prompts = [f"an H&E image of {c}." for c in CLASSES]
W = embed_texts(model, tokenizer, prompts)                    # (8, 512): one weight vector per class
print("classifier weights from text:", tuple(W.shape))

### 2.3 A tile is compared with every class vector, and the softmax picks a class: no training needed
These are the numbers in clip 2: BiomedCLIP gives the tumor tile 0.57 for colorectal adenocarcinoma.

In [ ]:
def zero_shot(model, z_image, W):
    # Class probabilities: cosine with each class vector, times the learned logit scale, softmax.
    return torch.softmax(model.logit_scale.exp().cpu() * (z_image @ W.T), dim=-1)

for name in ["tumor", "normal mucosa"]:
    p = zero_shot(model, embed_images(model, preprocess, [tile(CLIP_TILES[name])]), W)[0]
    print(f"{name} tile -> predicted '{CLASSES[int(p.argmax())]}'")
    for c, v in zip(CLASSES, p): print(f"    {c:>26s}  {v:.2f}")

The tumor tile is classified correctly, but not confidently. The normal mucosa tile is called adenocarcinoma: a
zero-shot classifier can be confidently wrong.

### 2.4 On ImageNet, CLIP's best model reaches 76.2% zero-shot
That result (ViT-L/14 at 336 pixels, Radford et al. 2021) matches a ResNet-50 trained on 1.28 million labeled images. The same
paper reports that zero-shot CLIP is weak on lymph node tumor detection (PatchCamelyon). We test this on our tiles.

### 2.5 On 5,000 H&E tiles in 8 classes, general CLIP scores 5.6 to 15.1%; chance is 12.5%
We embed all 5,000 tiles once with each model, then try three prompt wordings. This is the slow cell: a few minutes per
model on a CPU.

In [ ]:
TEMPLATES = ["an H&E image of {}.", "this is an image of {}.", "a histopathology image of {}."]
MODELS = {"CLIP ViT-B/32": "ViT-B-32/openai", "BiomedCLIP": BIOMEDCLIP}
features = {}                                                  # model name -> (5000, 512) image embeddings
zero_shot_acc = {}
for label, name in MODELS.items():
    t0 = time.time()
    m, pre, tok = load(name)
    features[label] = embed_images(m, pre, [Image.open(f).convert("RGB") for f in files])   # all 5,000 tiles
    zero_shot_acc[label] = []
    for t in TEMPLATES:
        W_t = embed_texts(m, tok, [t.format(c) for c in CLASSES])   # class vectors for this wording
        pred = (features[label] @ W_t.T).argmax(dim=1).numpy()  # the class with the highest cosine
        zero_shot_acc[label].append((pred == labels).mean())
        print(f"{label:>14s}  '{t}'  zero-shot accuracy {zero_shot_acc[label][-1]:.3f}")
    print(f"{label}: {time.time() - t0:.0f} s")
print("chance:", 1 / len(CLASSES))

### 2.6 BiomedCLIP, trained on biomedical figures, scores 34 to 39%: better, but far from reliable
Compare the two models and the three wordings. General CLIP's worst wording is **below chance** (12.5%). Changing only the
wording moves CLIP by almost 10 percentage points and BiomedCLIP by almost 5.

### 2.7 A linear probe with only 10 labels per class reaches 64% with CLIP and 77% with BiomedCLIP
This is the Week 6 linear probe on the same frozen embeddings: logistic regression trained on 1, 10, or 100 labeled tiles per
class and tested on all the other tiles, averaged over 5 random draws.

In [ ]:
probe_acc = {}
for label in MODELS:
    X = features[label].numpy()
    rng = np.random.default_rng(0)                             # same draws for both models
    probe_acc[label] = []
    for n in [1, 10, 100]:
        accs = []
        for s in range(5):
            idx = np.concatenate([rng.choice(np.where(labels == k)[0], n, replace=False) for k in range(len(CLASSES))])
            test = np.setdiff1d(np.arange(len(labels)), idx)   # every tile not used for training
            clf = LogisticRegression(max_iter=3000).fit(X[idx], labels[idx])
            accs.append((clf.predict(X[test]) == labels[test]).mean())
        probe_acc[label].append(np.mean(accs))
        print(f"{label:>14s}  linear probe, {n:>3d} labels per class: {probe_acc[label][-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
groups = [("CLIP\nzero-shot", zero_shot_acc["CLIP ViT-B/32"], "0.45"), ("BiomedCLIP\nzero-shot", zero_shot_acc["BiomedCLIP"], "#0F766E"),
          ("probe, 10 labels/class", [probe_acc["CLIP ViT-B/32"][1], probe_acc["BiomedCLIP"][1]], None)]
x = 0
for name, vals, color in groups:
    for k, v in enumerate(vals):
        c = color if color else ("0.45" if k == 0 else "#0F766E")
        ax.bar(x + k, 100 * v, color=c, alpha=1 if color is None else 1 - 0.3 * k)
        ax.text(x + k, 100 * v + 1, f"{100 * v:.1f}", ha="center", fontsize=8)
    ax.text(x + (len(vals) - 1) / 2, -14, name, ha="center", fontsize=8)
    x += len(vals) + 1
ax.axhline(12.5, ls="--", color="orange"); ax.text(x - 1, 14, "chance 12.5%", color="orange", fontsize=8, ha="right")
ax.set_xticks([]); ax.set_ylabel("accuracy (%)"); ax.set_ylim(0, 100)
plt.title("Kather 2016, 8 classes, 5,000 tiles"); plt.tight_layout(); plt.show()

**Reading the result.** Zero-shot classification is a quick baseline that needs no labels, but on tissue types it is weak and
sensitive to wording. A domain model (BiomedCLIP) helps. A handful of labels and a linear probe on the same embeddings helps
far more. Your numbers can differ slightly on a GPU.

## 3. Images as tokens: Med-PaLM M  *(clip 3)*

### 3.1 CLIP can match an image with a caption, but it cannot write a report
To write text about an image, the image has to go into a language model. Med-PaLM M (Tu et al., NEJM AI 2024) is not public,
so we use **SmolVLM-256M**, a small open model built the same way: a vision encoder, a projection, and a language model
decoder. First, a stylized chest X-ray like the one in the clip, drawn in code (no patient data).

In [ ]:
from matplotlib.patches import Ellipse, FancyBboxPatch, Rectangle, Arc

def stylized_cxr(size=4):
    # Draw a stylized frontal chest X-ray and return it as a PIL image (the same shapes as in clip 3).
    fig = plt.figure(figsize=(size * 0.86, size), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(-0.43, 0.43); ax.set_ylim(-0.5, 0.5); ax.axis("off")
    ax.add_patch(Rectangle((-0.43, -0.5), 0.86, 1.0, color="#1F2937"))   # film
    for s in (-1, 1):
        ax.add_patch(Ellipse((s * 0.172, 0.02), 0.29, 0.62, color="#0B0F17"))   # lungs
        for k in range(5):                                       # ribs
            ax.add_patch(Arc((s * 0.2, 0.2 - 0.1 * k), 0.36, 0.12, theta1=0 if s > 0 else 180, theta2=180 if s > 0 else 360, color="#9CA3AF", alpha=0.55, lw=2))
        ax.plot([s * 0.04, s * 0.31], [0.33, 0.38], color="#E5E7EB", lw=5)   # clavicles
        ax.add_patch(Arc((s * 0.2, -0.29), 0.34, 0.12, theta1=0, theta2=180, color="#E5E7EB", lw=4))   # diaphragm
    ax.add_patch(Rectangle((-0.37, -0.48), 0.74, 0.16, color="#D1D5DB", alpha=0.55))   # abdomen
    ax.add_patch(FancyBboxPatch((-0.039, -0.45), 0.078, 0.9, boxstyle="round,pad=0.005", color="#E5E7EB", alpha=0.85))   # spine
    ax.add_patch(Ellipse((0.043, -0.12), 0.33, 0.28, color="#D1D5DB", alpha=0.9))   # heart
    fig.canvas.draw()
    img = Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]); plt.close(fig)
    return img

xray = stylized_cxr()
plt.figure(figsize=(2.6, 3)); plt.imshow(xray); plt.axis("off"); plt.title("stylized chest X-ray", fontsize=9); plt.show()

### 3.2 A vision transformer turns the X-ray into patch vectors, and a projection resizes them to tokens
The processor resizes the image and cuts it into crops; the vision encoder turns each crop into patch vectors, and a projection
maps them to the size of the language model's word vectors. In the input, every image vector has a placeholder token
`<image>`. We count them, first for the image as one crop, then with the processor's default splitting into many crops.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
VLM = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained(VLM)
vlm = AutoModelForImageTextToText.from_pretrained(VLM, dtype=torch.float32).to(DEVICE).eval()
print(f"SmolVLM parameters: {sum(p.numel() for p in vlm.parameters()):,}")

messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Write the findings for this chest X-ray."}]}]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
IMAGE_ID = processor.tokenizer.convert_tokens_to_ids("<image>")
for split in (False, True):
    inputs = processor(text=prompt, images=[xray], return_tensors="pt", do_image_splitting=split)
    ids = inputs["input_ids"][0]
    print(f"splitting {str(split):>5s}: {inputs['pixel_values'].shape[1]:>2d} crop(s) -> "
          f"{int((ids == IMAGE_ID).sum()):>4d} image tokens, {len(ids) - int((ids == IMAGE_ID).sum())} text tokens")

Each crop becomes 64 image tokens. Splitting the image into more crops shows the model more detail, at the cost of many more
tokens: here the image takes far more of the sequence than the instruction does.

### 3.3 Image tokens and text tokens form one sequence for the same decoder as in Weeks 4 and 5
We decode the input with the image tokens collapsed, so the layout is visible, then let the model write.

In [ ]:
inputs = processor(text=prompt, images=[xray], return_tensors="pt", do_image_splitting=False).to(DEVICE)
text = processor.tokenizer.decode(inputs["input_ids"][0])
n_img = text.count("<image>")
print(text.replace("<image>" * n_img, f"<image> x {n_img}"))   # the sequence the decoder reads

with torch.no_grad():
    out = vlm.generate(**inputs, max_new_tokens=40, do_sample=False)
print("\nSmolVLM:", processor.tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))

The picture is a drawing and SmolVLM is tiny, so do not expect a useful report. Med-PaLM M was fine-tuned on real chest
X-rays with their reports, and its language model has up to 540 billion parameters.

### 3.4 Training uses the same next-token loss, counted only on the report tokens
As in Week 5, we append a report and set the labels of every image and instruction position to −100, so only the report tokens
count.

In [ ]:
report = " The lungs are clear. No pleural effusion.<end_of_utterance>"   # an illustrative report
full = processor(text=prompt + report, images=[xray], return_tensors="pt", do_image_splitting=False).to(DEVICE)
n_prompt = inputs["input_ids"].shape[1]                        # image + instruction tokens
labels_vlm = full["input_ids"].clone()
labels_vlm[:, :n_prompt] = -100                                # not counted
with torch.no_grad():
    loss = vlm(**full, labels=labels_vlm).loss
print(f"tokens: {full['input_ids'].shape[1]} in total, {int((labels_vlm != -100).sum())} counted in the loss")
print(f"next-token loss on the report: {loss.item():.2f}")

### 3.5 to 3.7 Med-PaLM M
Med-PaLM M fine-tunes PaLM-E (a PaLM language model joined to a vision transformer, in sizes 12B, 84B, and 562B) on
MultiMedBench: 14 tasks, 12 datasets, and 7 modalities, with one set of weights for all of them. Clinicians preferred its chest
X-ray report to the radiologist's in up to 40.5% of 246 cases, and it detected tuberculosis zero-shot with 87.7% accuracy,
against 92.6% for a specialist model. That was a retrospective, side-by-side ranking, not clinical use: in most cases the
model's report was not preferred. We return to evaluation in Week 12.

## Recap

| Clip | Code in this notebook |
|---|---|
| 1. CLIP | `embed_images`, `embed_texts`, the 4 × 4 grid, `clip_loss` (0.82 and 0.18) |
| 2. Zero-shot | prompts as class vectors, `zero_shot` (0.57), all 5,000 tiles with three wordings, the linear probe |
| 3. Images as tokens | `stylized_cxr`, counting `<image>` tokens, a loss counted only on the report |

> **Try it (optional):** write your own prompts in section 2.5, for example "a photo of {}" or longer descriptions of each
> tissue, and see how much the accuracy moves. In section 3.2, count the image tokens for an H&E tile instead of the X-ray.